# Lamplighter — hyperparameter sweeps, owning the loop

The **Optimize** tab runs sweeps visually; this notebook is the *eject path* —
the same mechanism as copyable code, for when you want to own the loop
(custom samplers, cross-study logic, anything the form doesn't model).

Either way, every trial is a REAL managed run: it streams to the open app
live, records into the Runs list under its study tag, and its snapshot shows
exactly the hyperparameters (and generated code) it trained with.

Requires the sweep extra: `pip install "lamplighter[sweep]"`.

> This notebook lives in `examples/`; the session cell puts the repo root on
> `sys.path` so `import lamplighter` resolves.

In [ ]:
import torch
from torchvision import datasets

mnist = datasets.MNIST(root="./data", train=True, download=True)
X = (mnist.data.float() / 255.0).view(-1, 784)
y = mnist.targets
torch.manual_seed(0)
idx = torch.randperm(len(X))[:8000]  # subsample for a snappy CPU sweep
X, y = X[idx], y[idx]
X.shape, y.shape

In [ ]:
import sys
from pathlib import Path

# The repo root (this notebook runs with examples/ as its cwd).
sys.path.insert(0, str(Path.cwd().parent.resolve()))

import lamplighter

# A per-notebook autosave path, so this canvas never clobbers another example's.
sess = lamplighter.Lamplighter(persist=".lamplighter/optuna.json")
sess.data(X=X, y=y)
sess.open()

## Set up the model

**Templates ▾ → MLP classifier** drops in a working 784 → 128 → 10 model. Then
on the **Models** canvas select the **Data** node, pick `X` / `y`, and set
**Validation Split** to `0.2` — the sweep minimizes `val_loss`, so it needs a
held-out split to judge by.

In [ ]:
# The sweep: lr (a training knob, merged into p.training) AND the hidden
# width (a NODE param — dict surgery on the trial's own graph copy). Each
# trial goes through the same run manager the ▶ Run button uses, so watch
# the app while this cell runs: trials stream live and land in the Runs list.
import optuna

from lamplighter.backend import state
from lamplighter.backend.runner import run_manager

project = state.get_project()  # the live canvas (same kernel as the app)


def objective(trial):
    p = project.model_copy(deep=True)
    p.training = {
        **(project.training or {}),
        "lr": trial.suggest_float("lr", 1e-4, 1e-1, log=True),
        "epochs": 6,
    }
    # The hidden layer = the first Linear in the graph (robust to node ids).
    hidden = next(n for n in p.models[0].graph.nodes if n.type == "Linear")
    hidden.params["out_features"] = trial.suggest_int("hidden", 32, 256)

    err = run_manager.start(p, source="notebook", study="nb-sweep")
    assert err is None, err
    run_manager.join()
    assert run_manager.state == "done", run_manager.error
    return run_manager.history["val_loss"][-1]


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=7),  # a seeded sweep is replayable
)
study.optimize(objective, n_trials=8)
print("best:", study.best_params, round(study.best_value, 4))

## What just landed in the app

Every trial is now a row in the **Runs** list (Training tab, side pane) —
recorded with `source: "notebook"` and the `nb-sweep` study tag, each with its
full curves and a reproducibility snapshot showing the exact lr/width it ran.
Click any row to view it on the dashboard, or **⊕ compare** a few — the diff
table shows which hyperparameters differed.

One thing the notebook path does NOT do automatically (the Optimize tab does):
keep the best trial's *weights* — the kernel only holds the most recent
trial's model. So finish by re-running the best config once and naming it:

In [ ]:
# The trials, ranked by their recorded final val_loss.
trials = [m for m in sess.checkpoints() if m.get("study") == "nb-sweep"]
for m in sorted(trials, key=lambda m: m["val_loss"] or float("inf"))[:5]:
    print(f"{m['name']:>8}  val {m['val_loss']:.4f}")

# Re-run the winner once and keep its weights under a name — now it's
# restorable/resumable from the app (and sess.model holds it here).
p = project.model_copy(deep=True)
p.training = {**(project.training or {}), "lr": study.best_params["lr"], "epochs": 6}
next(n for n in p.models[0].graph.nodes if n.type == "Linear").params["out_features"] = (
    study.best_params["hidden"]
)
assert run_manager.start(p, source="notebook", study="nb-sweep") is None
run_manager.join()
sess.checkpoint("nb-sweep-best")

## Tear down

Stops the server thread. (A kernel restart also stops it.)

In [ ]:
lamplighter.stop()